# Solution D1 · Reading the coefficients: units, standardization and the logarithm

**Type**: Interpret

## Objective

Answer questions about the effect of each variable in a salary model, showing that the largest coefficient
is not the most important variable, and translating the coefficients of a log-linear model into
percentages.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

df = ds.salaries()
X = df.drop(columns="salary")
y = df["salary"]
profile_a = pd.DataFrame({"years_experience": [10.0], "weekly_hours": [40], "big_city": [1], "degree": [1]})
profile_b = pd.DataFrame({"years_experience": [10.0], "weekly_hours": [40], "big_city": [0], "degree": [0]})
df.describe().round(1)

## Commented solution

In [ ]:
raw = LinearRegression().fit(X, y)
raw_coefs = pd.Series(raw.coef_, index=X.columns)
raw_most_influential = raw_coefs.abs().idxmax()
print("euros per unit of each feature:")
print(raw_coefs.round(1).to_string())
print("-> the largest coefficient:", raw_most_influential)

`big_city` wins because its "step" is a jump from 0 to 1: it is an effect of a few thousand euros for a whole
unit. `years_experience` is worth little per year, but the years vary over a range of 30 units. Comparing the
coefficients means comparing the effect of **one unit of each variable**: units that are not comparable.

In [ ]:
X_std = pd.DataFrame(StandardScaler().fit_transform(X), columns=X.columns)
std_coefs = pd.Series(LinearRegression().fit(X_std, y).coef_, index=X.columns)
std_most_influential = std_coefs.abs().idxmax()
print("euros per standard deviation of each feature:")
print(std_coefs.round(0).to_string())
print("-> the largest coefficient:", std_most_influential)

The model on the logarithm of the salary has coefficients that are **changes of the logarithm**: small numbers that
must be read as relative changes. To avoid mistakes you exponentiate: `exp(β) − 1` is the exact percentage
change, and for small β it is almost equal to β.

In [ ]:
log_model = LinearRegression().fit(X, np.log(y))
log_coefs = pd.Series(log_model.coef_, index=X.columns)
effect_pct = (np.exp(log_coefs) - 1) * 100
degree_effect_pct = effect_pct["degree"]
year_effect_pct = effect_pct["years_experience"]
print(pd.DataFrame({"coef (log)": log_coefs.round(4), "effect %": effect_pct.round(2)}).to_string())

In [ ]:
salary_a = np.exp(log_model.predict(profile_a))[0]
salary_b = np.exp(log_model.predict(profile_b))[0]
salary_ratio = salary_a / salary_b
ratio_from_coefs = np.exp(log_coefs["big_city"] + log_coefs["degree"])
print(f"salary A {salary_a:,.0f} | salary B {salary_b:,.0f} | ratio {salary_ratio:.3f} = exp(sum of the coefs) {ratio_from_coefs:.3f}")
print(f"difference in euros: {salary_a - salary_b:,.0f}")

In [ ]:
true_false = {"a": False, "b": True, "c": False}

## Check

In [ ]:
with verify():
    check(raw_most_influential == raw_coefs.abs().idxmax(), "variable with the largest raw coefficient identified",
          "use raw_coefs.abs().idxmax()")
    check(std_most_influential == std_coefs.abs().idxmax() and std_most_influential == "years_experience",
          "after standardization the most influential variable is a different one",
          "retrain on standardized features (StandardScaler) and look for the largest coefficient in absolute value")
    check(raw_most_influential != std_most_influential,
          "the largest coefficient depends on the unit of measure: the two answers are different",
          "the two 'most influential' variables should be different")
    check(7 < degree_effect_pct < 12, "plausible percentage effect of the degree",
          "degree_effect_pct should be about 9-10%: did you use np.exp(coef) - 1?")
    check(2.5 < year_effect_pct < 3.6, "plausible percentage effect of one year of experience",
          "year_effect_pct should be about 3%")
    check(close(salary_ratio, ratio_from_coefs), "the effect in the log model is multiplicative",
          "the ratio between the predicted salaries must match exp(coef_city + coef_degree)")
    check(true_false == {"a": False, "b": True, "c": False}, "statements judged correctly",
          "reread the meaning of the coefficients: they depend on the unit of measure, and in the log model the effects are multiplicative")
done()

## What you learned

- In the model on the raw target the coefficient of `big_city` (about 5,300 euros) is the largest in absolute value,
  but after standardization the most influential variable is `years_experience` (about 8,900 euros per standard
  deviation). The coefficients measure "per unit of each feature": to compare importance you need comparable
  units (standardized coefficients, or better still the effect of a realistic change).
- In the log model the effects are **multiplicative**: degree and big city combine with a product (1.09 · 1.16),
  not with a sum. And the effect in euros depends on the starting level: +9% on 20,000 is 1,800 euros, +9% on 60,000
  is 5,400.
- A coefficient of 0.0298 in the log model is ≈ 3%: the difference between 0.0298 and `exp(0.0298) - 1 = 3.03%` becomes
  relevant only for large coefficients (β = 0.5 is +65%, not +50%).

## Common pitfall

Saying "the variable with the largest coefficient is the most important". And its twin: standardizing a binary variable
and reading the result as the "effect of one standard deviation" makes little sense (for a 0/1 the standard
deviation depends on the proportion of ones). For binary variables it is preferable to read directly the effect of the step
from 0 to 1.

## Interview question

<details><summary>The coefficient of 'years of experience' in your model is 1,051 euros, the one of 'big city' is 5,348. The boss says 'the city matters five times more than experience'. What do you answer?</summary>

That the two numbers are not comparable: 1,051 euros is the effect of **one year** of experience, 5,348 that of the jump
from "not a big city" to "big city". To compare importance I would use standardized coefficients or a concrete question:
how much does the predicted salary change between an employee at the 10th and one at the 90th percentile of experience,
versus changing city? In this dataset experience, over a realistic range, weighs more than the city.

</details>